# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source

df = pd.read_csv('/Users/happy/Downloads/global_energy_mix.csv')


# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [5]:
# Task 1
# YOUR CODE HERE
# ── Task 1: Treemap of fossil fuel TWh ─────────────────────────────
VALUE_COL = 'TWh'   # change this to the name of your CSV's generation column

# One fixed colour per fossil source. All three pass a colour-blindness check:
# the closest pair, gas vs oil, still separates clearly for deutan (red-green) vision.
# Plotly labels parent tiles '(?)' when colour is categorical,
# so mapping '(?)' to grey makes the Region and Country tiles grey.
FOSSIL_COLORS = {
    'Coal':        '#2a78d6',   # blue
    'Oil':         '#eb6834',   # orange
    'Natural Gas': '#1baf7a',   # aqua
    '(?)':         '#e4e3df',   # grey parent tiles
}

# If the file covers several years, keep only the latest so TWh isn't summed across years
if 'Year' in df.columns:
    df_t1 = df[df['Year'] == df['Year'].max()]
else:
    df_t1 = df

# 1. Filter to fossil sources only
fossil = (df_t1.loc[df_t1['Source_Type'] == 'Fossil']
               .groupby(['Region', 'Country', 'Source'], as_index=False)[VALUE_COL].sum())

# 2. Work out the insight from the data instead of hard-coding it
region_share = (fossil.groupby('Region')[VALUE_COL].sum()
                / df_t1.groupby('Region')[VALUE_COL].sum()).sort_values(ascending=False)
top_region, top_share = region_share.index[0], region_share.iloc[0]

country_fossil = fossil.groupby('Country')[VALUE_COL].sum().sort_values(ascending=False)
top_country, top_twh = country_fossil.index[0], country_fossil.iloc[0]

title = (f"<b>{top_region} is the most fossil-dependent region "
         f"({top_share:.0%} of its generation)</b><br>"
         f"<sup>{top_country} burns the most fossil fuel at {top_twh:,.0f} TWh · "
         f"fossil generation only, by Region → Country → Source</sup>")

# 3. Plot
fig = px.treemap(
    fossil,
    path=['Region', 'Country', 'Source'],
    values=VALUE_COL,
    color='Source',
    color_discrete_map=FOSSIL_COLORS,
)
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',        # TWh only, no percentages
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<br>%{id}<extra></extra>',
    textfont=dict(color='#1a1a19'),                       # dark text is readable on all three colours
    marker=dict(line=dict(color='#fcfcfb', width=2)),     # thin gaps between tiles
    root_color='#fcfcfb',
    tiling=dict(pad=3),
)
fig.update_layout(
    title=dict(text=title, x=0.01),
    margin=dict(t=80, l=10, r=10, b=10),
    paper_bgcolor='#fcfcfb',
    font=dict(family='Inter, Arial, sans-serif', color='#1a1a19'),
)
fig.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [6]:
# Task 2 — charts
# YOUR CODE HERE
# ── Task 2: Low-carbon TWh by country, treemap vs bar ───────────────
LOWCARB_COLOR = '#2a78d6'   # one series, so one colour; blue stays distinct for colour-blind viewers

df_t2 = df[df['Year'] == df['Year'].max()] if 'Year' in df.columns else df

lowcarb = (df_t2.loc[df_t2['Source_Type'] == 'Low-carbon']
                .groupby('Country', as_index=False)[VALUE_COL].sum()
                .sort_values(VALUE_COL, ascending=False))
lowcarb['All'] = 'Low-carbon'          # dummy root node

# Treemap
fig_tree = px.treemap(
    lowcarb,
    path=['All', 'Country'],
    values=VALUE_COL,
    color='All',                                      # every tile gets the same colour value...
    color_discrete_map={'Low-carbon': LOWCARB_COLOR}, # ...so plotly doesn't cycle through shades
)
fig_tree.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    textfont=dict(color='white'),
    marker=dict(line=dict(color='#fcfcfb', width=2)),
    root_color='#fcfcfb',
    tiling=dict(pad=3),
)
fig_tree.update_layout(
    title=dict(text='<b>Low-carbon generation (Nuclear + Hydro) by country</b>', x=0.01),
    margin=dict(t=60, l=10, r=10, b=10),
    paper_bgcolor='#fcfcfb',
    font=dict(family='Inter, Arial, sans-serif', color='#1a1a19'),
)
fig_tree.show()

# Horizontal bar
top = lowcarb.iloc[0]
runner_up = lowcarb.iloc[1]
bar_title = (f"<b>{top['Country']} leads low-carbon generation with {top[VALUE_COL]:,.0f} TWh</b><br>"
             f"<sup>{top[VALUE_COL] - runner_up[VALUE_COL]:,.0f} TWh ahead of {runner_up['Country']} · Nuclear + Hydro</sup>")

fig_bar = px.bar(
    lowcarb,
    x=VALUE_COL, y='Country',
    orientation='h',
    text=VALUE_COL,
    color_discrete_sequence=[LOWCARB_COLOR],
)
fig_bar.update_traces(
    texttemplate='%{x:,.0f} TWh',
    textposition='outside',
    textfont=dict(color='#1a1a19'),
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} TWh<extra></extra>',
    marker_cornerradius=4,          # needs plotly ≥ 5.19; delete this line if it errors
    cliponaxis=False,
)
fig_bar.update_layout(
    title=dict(text=bar_title, x=0.01),
    yaxis=dict(categoryorder='total ascending', title=None),   # largest bar at the top
    xaxis=dict(title='TWh', showgrid=True, gridcolor='#ebeae6', zeroline=False),
    bargap=0.35,
    plot_bgcolor='#fcfcfb', paper_bgcolor='#fcfcfb',
    margin=dict(t=80, l=10, r=60, b=40),
    font=dict(family='Inter, Arial, sans-serif', color='#1a1a19'),
)
fig_bar.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*YOUR ANSWER HERE*

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [7]:
# Task 3
# YOUR CODE HERE
# ── Task 3: 100% stacked bar, coal share of each region's fossil mix ──
df_fossil = fossil      # Task 1 named this `fossil`; alias it to the name the brief uses

SOURCE_ORDER = ['Coal', 'Oil', 'Natural Gas']       # Coal first = sits on the baseline
STACK_COLORS = {
    'Coal':        '#2a78d6',   # same blue as Task 1, the only coloured segment
    'Oil':         '#c3c2b7',   # mid grey
    'Natural Gas': '#e4e3df',   # light grey
}

region_mix = df_fossil.groupby(['Region', 'Source'], as_index=False)[VALUE_COL].sum()
region_mix['Share'] = region_mix[VALUE_COL] / region_mix.groupby('Region')[VALUE_COL].transform('sum')

# Sort regions by coal share, highest first
coal_share = (region_mix.loc[region_mix['Source'] == 'Coal']
                        .set_index('Region')['Share']
                        .reindex(region_mix['Region'].unique(), fill_value=0)   # regions with no coal = 0
                        .sort_values(ascending=False))
region_order = coal_share.index.tolist()
top_region, top_coal = region_order[0], coal_share.iloc[0]

# Labels on coal segments only
region_mix['Label'] = region_mix['Share'].map('{:.0%}'.format).where(region_mix['Source'] == 'Coal', '')

title = (f"<b>{top_region} is the most coal-dependent region: "
         f"coal is {top_coal:.0%} of its fossil generation</b><br>"
         f"<sup>Share of each region's fossil TWh by source · sorted by coal share</sup>")

fig_stack = px.bar(
    region_mix,
    x='Share', y='Region',
    color='Source',
    orientation='h',
    text='Label',
    custom_data=['Source', VALUE_COL],
    category_orders={'Source': SOURCE_ORDER, 'Region': region_order},   # first region = top bar
    color_discrete_map=STACK_COLORS,
)
fig_stack.update_traces(
    textposition='inside',
    insidetextanchor='start',
    textfont=dict(color='white'),
    marker_line=dict(color='#fcfcfb', width=2),      # thin gap between segments
    hovertemplate='<b>%{y}</b> · %{customdata[0]}<br>%{x:.0%} of fossil · %{customdata[1]:,.0f} TWh<extra></extra>',
)
fig_stack.update_layout(
    title=dict(text=title, x=0.01),
    barmode='stack',
    xaxis=dict(title=None, tickformat='.0%', range=[0, 1], showgrid=True, gridcolor='#ebeae6', zeroline=False),
    yaxis=dict(title=None),
    legend=dict(title=None, orientation='h', yanchor='bottom', y=1.0, xanchor='left', x=0, traceorder='normal'),
    bargap=0.35,
    plot_bgcolor='#fcfcfb', paper_bgcolor='#fcfcfb',
    margin=dict(t=110, l=10, r=20, b=40),
    font=dict(family='Inter, Arial, sans-serif', color='#1a1a19'),
)
fig_stack.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

*YOUR ANSWER HERE*: “Which countries burn the most fossil fuel, and how big is each one’s contribution to the total?” Tile area is proportional to absolute TWh, and the Region → Country → Source hierarchy is kept, so you can see that one large country can make up most of its region’s fossil use. The stacked bar can’t answer this. It shows only regions, and every bar is stretched to 100%, so a region that burns ten times more fossil fuel than another gets a bar of the same length.